# arXiv abstracts -> Chroma vector store (LangChain)

Topics: **ML, DL, NLP, Computer Vision, LLMs**.

Pipeline: read abstracts from the **Kaggle arXiv metadata snapshot** -> filter by category and date -> build LangChain `Document`s with filterable metadata -> embed (local HuggingFace model) -> persist in Chroma -> retrieve with filters + similarity -> answer (done in rag_pipeline laterr).


In [ ]:
import os, re, json, shutil
from pathlib import Path
from datetime import datetime, date
from email.utils import parsedate_to_datetime

from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()


DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
RAW_FILE = DATA_DIR / "arxiv-metadata-oai-snapshot.json"   # the Kaggle file (optional manual location)
FILTERED_FILE = DATA_DIR / "arxiv_filtered.jsonl"          # our filtered cache
CHROMA_DIR = "chroma_arxiv"

# embeddings
EMBEDDINGS_PROVIDER = "huggingface"     
HF_MODEL = "BAAI/bge-small-en-v1.5"     
OPENAI_MODEL = "text-embedding-3-small"  # not gonna use it
# separate collection per provider, because vectors from different models must not be mixed
COLLECTION = f"arxiv_{EMBEDDINGS_PROVIDER}"

if EMBEDDINGS_PROVIDER == "openai":
    assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in a .env file next to this notebook"

#  what to keep from the snapshot 
CATEGORIES = {
    "cs.LG":   "Machine Learning",
    "stat.ML": "Machine Learning (stat)",
    "cs.AI":   "Artificial Intelligence",
    "cs.NE":   "Neural and Evolutionary Computing (deep learning)",
    "cs.CL":   "Computation and Language (NLP / LLMs)",
    "cs.CV":   "Computer Vision",
}
DATE_FROM = "2024-01-01"      # we dont need the whole dataset, just  2024 and after
MAX_PER_CATEGORY = 5000       # there are 100k+ matching papers so cap it
REFRESH = False               

## 1. Get the Kaggle dataset
Dataset: [Cornell-University/arxiv](https://www.kaggle.com/datasets/Cornell-University/arxiv) - one big JSON-lines file (`arxiv-metadata-oai-snapshot.json`, several GB) with the metadata and abstract of every arXiv paper.

(the code below uses `kagglehub`): needs a free Kaggle account and credentials, either `~/.kaggle/kaggle.json` or `KAGGLE_USERNAME` / `KAGGLE_KEY` in your `.env`.

The raw file is only read once; the filtered result is cached in `data/arxiv_filtered.jsonl`.

In [ ]:
def ensure_raw_file():
    if RAW_FILE.exists():
        return RAW_FILE
    try:
        import kagglehub
        folder = Path(kagglehub.dataset_download("Cornell-University/arxiv"))
        return next(folder.rglob("arxiv-metadata-oai-snapshot.json"))
    except Exception as e:
        raise FileNotFoundError(
            "Could not download the Kaggle dataset automatically "
            f"({type(e).__name__}: {e}).\n"
            "Download it manually from https://www.kaggle.com/datasets/Cornell-University/arxiv "
            f"and place arxiv-metadata-oai-snapshot.json at {RAW_FILE}"
        )

def clean(s):
    return re.sub(r"\s+", " ", s or "").strip()

def first_submitted(rec):
    # date of the first version (the real "published" date); fall back to update_date
    try:
        return parsedate_to_datetime(rec["versions"][0]["created"]).date()
    except Exception:
        return date.fromisoformat(rec["update_date"])

def authors_list(rec, limit=None):
    parsed = rec.get("authors_parsed") or []
    names = [f"{a[1]} {a[0]}".strip() for a in parsed]
    return names if limit is None else names[:limit]

def load_filtered():
    if FILTERED_FILE.exists() and not REFRESH:
        with open(FILTERED_FILE, encoding="utf-8") as f:
            return [json.loads(line) for line in f]

    raw = ensure_raw_file()
    wanted = set(CATEGORIES)
    start = date.fromisoformat(DATE_FROM)
    buckets = {c: [] for c in CATEGORIES}

    with open(raw, encoding="utf-8") as f:
        for line in tqdm(f, desc="scanning snapshot"):
            rec = json.loads(line)
            cats = rec["categories"].split()
            hit = [c for c in cats if c in wanted]
            if not hit:
                continue
            published = first_submitted(rec)
            if published < start:
                continue
            abstract = clean(rec["abstract"])
            if not abstract or abstract.lower().startswith("this paper has been withdrawn"):
                continue
            primary = cats[0] if cats[0] in wanted else hit[0]
            buckets[primary].append({
                "arxiv_id": rec["id"],
                "title": clean(rec["title"]),
                "abstract": abstract,
                "authors": authors_list(rec, limit=6),
                "categories": cats,
                "primary_category": cats[0],
                "published": published.isoformat(),
                "url": f"https://arxiv.org/abs/{rec['id']}",
                "pdf_url": f"https://arxiv.org/pdf/{rec['id']}",
            })

    papers = []
    for c, items in buckets.items():
        items.sort(key=lambda p: p["published"], reverse=True)
        papers.extend(items[:MAX_PER_CATEGORY])
        print(f"{c}: kept {min(len(items), MAX_PER_CATEGORY)} of {len(items)}")

    with open(FILTERED_FILE, "w", encoding="utf-8") as f:
        for p in papers:
            f.write(json.dumps(p) + "\n")
    return papers

papers = load_filtered()
print(f"{len(papers)} papers")

## 2. Build LangChain Documents with filterable metadata
`page_content` is the abstract (this is what gets embedded). Everything you want to filter or cite on goes in `metadata`.

In [ ]:
from langchain_core.documents import Document

# one boolean flag per category, e.g. "cs.CL" -> "cat_cs_CL"
CAT_FLAGS = {c: "cat_" + re.sub(r"[^A-Za-z0-9]", "_", c) for c in CATEGORIES}
LLM_RE = re.compile(r"\b(large language models?|LLMs?)\b", re.I)

def to_int_date(d):
    # accepts "2024-06-01", "20240601", datetime or date -> 20240601
    if isinstance(d, (datetime, date)):
        return int(d.strftime("%Y%m%d"))
    return int(re.sub(r"\D", "", str(d))[:8])

def to_document(p):
    dt = date.fromisoformat(p["published"])
    authors = ", ".join(p["authors"][:5]) + (" et al." if len(p["authors"]) > 5 else "")
    meta = {
        "arxiv_id": p["arxiv_id"],
        "title": p["title"],
        "authors": authors,
        "primary_category": p["primary_category"],
        "categories": " ".join(p["categories"]),        # for display (all categories)
        "published": p["published"],                    #for display
        "published_ts": to_int_date(dt),                # for range filters (YYYYMMDD)
        "year": dt.year,
        "url": p["url"],
        "pdf_url": p["pdf_url"],
        "mentions_llm": bool(LLM_RE.search(p["title"] + " " + p["abstract"])),
    }
    for cat, flag in CAT_FLAGS.items():                 # for category filters
        meta[flag] = cat in p["categories"]
    return Document(page_content=p["abstract"], metadata=meta)

docs = [to_document(p) for p in papers]
print(len(docs), "documents")
docs[0].metadata

## 3. Embed and store in Chroma
The same embeddings object is reused for queries later, so docs and queries live in the same vector space. Cosine distance is used. Ids are the arXiv ids, so re-running skips papers that are already indexed.

With the local HuggingFace model the first run downloads the model (~130 MB) and embeds on your CPU/GPU; expect several minutes per ~10k abstracts on CPU.

In [ ]:
from langchain_chroma import Chroma

if EMBEDDINGS_PROVIDER == "openai":
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model=OPENAI_MODEL)
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name=HF_MODEL,
        encode_kwargs={"normalize_embeddings": True},
    )

vectorstore = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)

existing = set(vectorstore.get(include=[])["ids"])
new_docs = [d for d in docs if d.metadata["arxiv_id"] not in existing]
print(f"{len(existing)} already indexed, {len(new_docs)} to add")

BATCH = 200
for i in tqdm(range(0, len(new_docs), BATCH), desc="embedding + indexing"):
    batch = new_docs[i:i + BATCH]
    vectorstore.add_documents(batch, ids=[d.metadata["arxiv_id"] for d in batch])

print("collection size:", vectorstore._collection.count())

## 4. Retrieval with filters + similarity
build_filter translates friendly arguments into a Chroma (where) dict:



In [5]:
def build_filter(categories=None, match="any", date_from=None, date_to=None,
                 year=None, llm_only=False):
    clauses = []
    if categories:
        flags = [{CAT_FLAGS[c]: {"$eq": True}} for c in categories]
        if len(flags) == 1:
            clauses.append(flags[0])
        else:
            clauses.append({"$or" if match == "any" else "$and": flags})
    if date_from:
        clauses.append({"published_ts": {"$gte": to_int_date(date_from)}})
    if date_to:
        clauses.append({"published_ts": {"$lte": to_int_date(date_to)}})
    if year:
        clauses.append({"year": {"$eq": int(year)}})
    if llm_only:
        clauses.append({"mentions_llm": {"$eq": True}})

    if not clauses:
        return None
    return clauses[0] if len(clauses) == 1 else {"$and": clauses}


def search(query, k=5, **filters):
    # returns [(Document, similarity)] with similarity = 1 - cosine distance
    results = vectorstore.similarity_search_with_score(query, k=k, filter=build_filter(**filters))
    return [(doc, 1 - dist) for doc, dist in results]


def citation(doc, n=None):
    m = doc.metadata
    prefix = f"[{n}] " if n else ""
    return f'{prefix}{m["authors"]} ({m["year"]}). "{m["title"]}". arXiv:{m["arxiv_id"]}. {m["url"]}'


def show(results):
    for i, (doc, sim) in enumerate(results, 1):
        m = doc.metadata
        print(f'{i}. {m["title"]}  (sim={sim:.3f})')
        print(f'   {m["published"]} | {m["categories"]}')
        print(f'   {doc.page_content[:200]}...\n')

In [ ]:
# plain similarity search
show(search("retrieval augmented generation for question answering", k=3))

# filtered: NLP + vision papers (either), from 2025 onwards
show(search("multimodal vision-language models", k=3,
            categories=["cs.CV", "cs.CL"], match="any", date_from="2025-01-01"))

# filtered: must be in BOTH cs.CL and cs.LG, within a date window, LLM-related
show(search("efficient fine-tuning of large models", k=3,
            categories=["cs.CL", "cs.LG"], match="all",
            date_from="2025-01-01", date_to="2025-12-31", llm_only=True))

## 5. Retriever + answers with citations
as_retriever plugs into any LangChain chain. The "Sources" list is built from metadata in code, not by the LLM, so citations always point to real retrieved papers.

Everything above works without any LLM. Only answer() below needs one: set "ollama" (local; install Ollama and run ollama pull llama3.1).

In [ ]:
# a LangChain retriever with a filter together
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5, "filter": build_filter(categories=["cs.CV"], date_from="2025-01-01")},
)
retriever.invoke("diffusion models for image generation")[:2]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

LLM_PROVIDER = "ollama"   # or "ollama"

def get_llm():
    if LLM_PROVIDER == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model="llama3.1", temperature=0)
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model="gpt-4o-mini", temperature=0)

llm = get_llm()

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using ONLY the numbered paper abstracts below. "
     "Cite supporting papers inline like [1], [2]. If the abstracts do not contain "
     "the answer, say so.\n\n{context}"),
    ("human", "{question}"),
])

def answer(question, k=5, **filters):
    results = search(question, k=k, **filters)
    if not results:
        return "No papers matched those filters."
    docs_ = [d for d, _ in results]
    context = "\n\n".join(
        f'[{i}] {d.metadata["title"]} ({d.metadata["published"]})\n{d.page_content}'
        for i, d in enumerate(docs_, 1)
    )
    reply = (prompt | llm).invoke({"context": context, "question": question}).content
    sources = "\n".join(citation(d, i) for i, d in enumerate(docs_, 1))
    return f"{reply}\n\nSources:\n{sources}"

print(answer("What are recent approaches to reduce hallucinations in LLMs?",
             categories=["cs.CL"], date_from="2025-01-01"))

In [ ]:
import numpy as np

info = {"provider": EMBEDDINGS_PROVIDER}

# model facts
q = embeddings.embed_query("test query")
info["embedding_dimension"] = len(q)
info["query_vector_norm"] = round(float(np.linalg.norm(q)), 4)   # ~1.0 => normalized (cosine == dot product)

if EMBEDDINGS_PROVIDER == "huggingface":
    st = embeddings._client                                      # the SentenceTransformer
    info["model"] = HF_MODEL
    info["max_input_tokens"] = st.max_seq_length                 # longer text gets truncated
    info["device"] = str(st.device)
    tok_len = lambda t: len(st.tokenizer.encode(t))
else:
    import tiktoken
    info["model"] = OPENAI_MODEL
    info["max_input_tokens"] = 8191
    enc = tiktoken.get_encoding("cl100k_base")
    tok_len = lambda t: len(enc.encode(t))

#  do the abstracts fit in the model's input window?             (makes sure it does)
sample = docs[:2000]
lens = np.array([tok_len(d.page_content) for d in sample])
info["abstract_tokens_mean"] = int(lens.mean())
info["abstract_tokens_p95"] = int(np.percentile(lens, 95))
info["abstract_tokens_max"] = int(lens.max())
info["pct_abstracts_truncated"] = round(float((lens > info["max_input_tokens"]).mean() * 100), 2)

# stored collection ------------------------------------------------------------------------------------------------
col = vectorstore._collection
row = col.get(limit=1, include=["embeddings", "metadatas"])
info["collection"] = col.name
info["documents_in_store"] = col.count()
info["distance_metric"] = (col.metadata or {}).get("hnsw:space", "l2 (Chroma default)")
info["stored_vector_dimension"] = len(row["embeddings"][0])
info["stored_matches_model"] = info["stored_vector_dimension"] == info["embedding_dimension"]
info["metadata_keys"] = sorted(row["metadatas"][0].keys())

for k, v in info.items():
    print(f"{k:28} {v}")

if EMBEDDINGS_PROVIDER == "huggingface" and "bge" in HF_MODEL.lower():
    print("\nnote: bge models work best when short queries are prefixed with "
          "'Represent this sentence for searching relevant passages: ' (documents get no prefix)")